#**LLM Project - Document Based Question Answering Chatbot**
###**Submitted by - Gaurav Dey**
###**Roll-24MDA029**

###**Install Required Libraries**

In [23]:
!pip install -q \
 pypdf \
 langchain \
 langchain-google-genai \
 "langchain-community<0.4.0" \
 langchain-google-vertexai \
 langchain-chroma \
 langchain-text-splitters \
 chromadb \
 ragas \
 datasets

###**Manually Restart Session**

###**Import the Libraries**

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata, drive

###**Set the Environment Variables**

In [2]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]="Document Based Q & A chatbot"
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

###**Check if LangSmith is working**

In [3]:
from langsmith import Client
client = Client()

###**Set Up Vector Database and Embeddings**

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0)

embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001")

vector_db = Chroma("QnAChatbot_info", embeddings_model)

###**Load the PDF Document from drive, chunk it and add the chunks to ChromaDB**

**Mount your Google Drive.**

In [6]:
drive.mount('/content/drive')

Mounted at /content/drive


**Load the PDF, chunk it and add it to the vector database.**

In [7]:
file_path = "/content/drive/MyDrive/Colab Notebooks/Sem 3/LLM Project/data/AI_Primer.pdf"
pdf_loader = PyPDFLoader(file_path)
pdf_chunks = text_splitter.split_documents(pdf_loader.load())
vector_db.add_documents(pdf_chunks)

['834fd729-e9fa-4fcc-98ee-e3106c98a744',
 'a897dad5-8b5a-45b2-ae6c-eb312ce00993',
 '9504617c-1bef-4cd5-9755-b1099dac674a',
 'a5ffe9f4-45e2-4998-b6ae-8c4e9f5cd458',
 '5e2ffd86-31c0-4eaa-800c-f9a96ccc19ad',
 'd28e29a7-7611-43d9-bfe1-3b5d4a9d2de9',
 'b0bc633a-ff63-4557-87a8-cf8c87c7e14f',
 '53ad1b8b-1d6f-4d63-8009-9ad044a81263',
 '30962c79-48e3-4ca0-821a-5aeb847ca042',
 '51950953-a83a-4b17-b392-73ff1eeb8358',
 'd40a0189-e365-4020-917d-b4cfc1133d97',
 '21795bf7-b783-49e3-b1d6-81a943310eb4',
 '870d20f9-e21c-48af-a1bf-714ffba9497a',
 '083c83cc-77fa-4395-8fa5-e05bc9c39a59',
 '3de25c3e-0b86-4604-899a-1e4fce930073',
 'e06469f4-115f-469b-aecf-7545d93281a3',
 '522a8411-32f6-48fa-b4f8-1c34282db20a',
 '7c1a9a01-be1d-414b-9066-5ec66ea863e4',
 '2698d8ac-9fb7-49b4-9a0a-b75b7f4dfdb2',
 '0af92abd-ac06-4ed1-981b-a3c140d7400d',
 'a9f638f9-358d-4da0-a37f-4c2390f07efc',
 '92c4e68e-cffb-435a-87de-57ef931db59e',
 'a58de7aa-ceb5-4cef-b217-1e8049f408c0',
 '345f31ad-ea32-4722-97e8-b95be4ea99ee',
 '9a104c0f-4366-

###**Querying the vector store directly**

In [8]:
query = "Describe two distinct real-world applications of AI in Healthcare."
results = vector_db.similarity_search(query, 4) # four clostest results

for i, doc in enumerate(results, 1):
    print(f"Result {i}:\n{doc.page_content}\n" + "-"*40)

Result 1:
Real-World Industry Applications
AI as a Transformative General-Purpose Technology
Artificial Intelligence operates as a foundational technology across virtually every global sector, driving operational
efficiency, automated decision support, and new product innovations.
Industry Sector
Primary AI Use Cases
Measurable Benefits & Value
Healthcare & Medicine
 Diagnostic Radiology (X-Ray/MRI analysis)
 AI Drug Discovery & Protein Folding
 Personalized Precision Medicine
----------------------------------------
Result 2:
 Healthcare leverage: AI speeds diagnostic imaging and assists in protein molecular drug discovery.
 Finance leverage: Real-time fraud detection algorithms protect digital transactions globally.
 Retail & Education leverage: Recommendation and adaptive platforms deliver highly personalized experiences.
A Beginner's Guide to AI, ML & Neural Networks
Page 9 of 10
Artificial Intelligence & Sub-Branches Handbook
----------------------------------------
Result 3

In [ ]:
len(results)

4

###**Sample Questions**

1.   What is the fundamental difference between Artificial Narrow AI (ANI) and Artificial General AI (AGI)?

2.   What were "AI Winters" in the historical development of AI,

3.   What three key factors drove the modern Deep Learning boom starting in the 2010s?

4.   Differentiate between Overfitting and Underfitting.

5.   Compare Supervised Learning and Unsupervised Learning.

6.   Provide one concrete real-world example for a classification task and one for a clustering task.

7.   Explain how an artificial neuron computes its output.

8.   What is the role of an activation function (e.g., ReLU or Softmax)?

9.   How does Object Detection differ from Semantic Segmentation in Computer Vision?

10. In Natural Language Processing (NLP), what are word embeddings?

11. What is Retrieval-Augmented Generation (RAG), and how does it improve the reliability of Large Language Models (LLMs)?

12. Describe two distinct real-world applications of AI in Healthcare.

13. What is SLAM (Simultaneous Localization and Mapping), and why is it essential for autonomous vehicles and mobile robots?

14. What causes Algorithmic Bias in machine learning, and why is the Human-in-the-Loop (HITL) framework emphasized for ethical AI deployment?

###**Asking a question through a RAG chain**

In [10]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """Use the following pieces of context
to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences at maximum and keep the answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

In [11]:
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

retriever = vector_db.as_retriever()

question_feeder = RunnablePassthrough()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

rag_chain = {"context": retriever,
     "question": question_feeder}|rag_prompt|llm

#utility function to execute the chain:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [12]:
question = """What were "AI Winters" in the historical development of AI, and what three key factors drove the modern Deep Learning boom starting in the 2010s?"""
answer = execute_chain(rag_chain, question)
print(answer)

content='AI Winters were cyclic periods of optimism and decline in AI development, occurring when inflated expectations outpaced hardware memory and computing limitations, leading to funding slowdowns. The modern Deep Learning boom, starting around 2010, was driven by three converging factors. These factors were Big Data, GPU Acceleration, and Deep Learning Breakthroughs.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a022f2-acf5-7240-9026-5e77b956a9a0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1371, 'output_tokens': 665, 'total_tokens': 2036, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 598}}


In [13]:
question = """Differentiate between Overfitting and Underfitting. How does partitioning data into Training, Validation, and Test sets help ensure proper generalization?"""
answer = execute_chain(rag_chain, question)
print(answer)

content='Overfitting occurs when a model memorizes training data noise and performs poorly on new data, while underfitting describes an overly simplistic model that fails to capture underlying patterns and performs poorly on all data. To ensure proper generalization, datasets are partitioned into Training, Validation, and Test sets. This split allows the model to learn from the Training set, be tuned using the Validation set, and finally be evaluated on unseen Test data to assess its true accuracy and prevent memorization.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a022f2-eebf-76e2-b6f6-d56a6376c641-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1350, 'output_tokens': 950, 'total_tokens': 2300, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 857}}


In [14]:
question = """Compare Supervised Learning and Unsupervised Learning. Provide one concrete real-world example for a classification task and one for a clustering task."""
answer = execute_chain(rag_chain, question)
print(answer.content)

Supervised Learning uses labeled data to predict outcomes, such as classifying emails as 'Spam' or 'Not Spam'. Unsupervised Learning works with unlabeled data to discover hidden structures, for example, grouping customers into segments (clustering).


In [15]:
question = """Explain how an artificial neuron computes its output. What is the role of an activation function (e.g., ReLU or Softmax)?"""
answer = execute_chain(rag_chain, question)
print(answer.content)

An artificial neuron computes its output by receiving numerical inputs, multiplying them by adjustable weights, adding a bias, and then passing the sum through an activation function. Activation functions introduce non-linearity into the network, enabling it to learn complex patterns. Common examples include ReLU, Sigmoid for probabilities, and Softmax for multi-class distribution.


###**Rag with Context Memory**

In [16]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

rag_prompt = ChatPromptTemplate.from_messages(
    [
      ("system", """You are a helpful assistant, world-class
          expert in Guwahati and Assam information. Provide interesting insights
          on local history and recommend places to visit with
          knowledgeable and engaging answers. Answer all questions
          to the best of your ability, but only use what has been
          provided in the context. If you don't know, just say you
          don't know. Use three sentences maximum and keep
          the answer as concise as possible."""),
      ("placeholder", "{chat_history_messages}"),
      ("assistant", "{retrieved_context}"),
      ("human", "{question}"),
    ])

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
chat_history_memory = ChatMessageHistory()

def get_messages(x):
    return chat_history_memory.messages

rag_chain = {
     "retrieved_context": retriever,
     "question": question_feeder,
     "chat_history_messages": RunnableLambda(get_messages)
    } | rag_prompt | chatbot

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')
    return answer

In [17]:
question = """How does Object Detection differ from Semantic Segmentation in Computer Vision? In Natural Language Processing (NLP), what are word embeddings?"""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='How does Object Detection differ from Semantic Segmentation in Computer Vision? In Natural Language Processing (NLP), what are word embeddings?', additional_kwargs={}, response_metadata={}), AIMessage(content="In Computer Vision, Object Detection identifies where specific objects are within an image, like locating pedestrians or cars. Semantic Segmentation, on the other hand, classifies every individual pixel in an image into a category, such as highlighting all road surface pixels.\n\nIn Natural Language Processing (NLP), word embeddings convert text tokens into continuous numerical vectors in a high-dimensional vector space. Words with similar meanings, such as 'king' and 'queen', share geometric proximity in this vector space.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a022f3-63a9-77f1-b22c-5058602e908a-0'

In [18]:
question = """What is Retrieval-Augmented Generation (RAG), and how does it improve the reliability of Large Language Models (LLMs)?"""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='How does Object Detection differ from Semantic Segmentation in Computer Vision? In Natural Language Processing (NLP), what are word embeddings?', additional_kwargs={}, response_metadata={}), AIMessage(content="In Computer Vision, Object Detection identifies where specific objects are within an image, like locating pedestrians or cars. Semantic Segmentation, on the other hand, classifies every individual pixel in an image into a category, such as highlighting all road surface pixels.\n\nIn Natural Language Processing (NLP), word embeddings convert text tokens into continuous numerical vectors in a high-dimensional vector space. Words with similar meanings, such as 'king' and 'queen', share geometric proximity in this vector space.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a022f3-63a9-77f1-b22c-5058602e908a-0'

###**Evaluation**

We can evaluate the RAG application using Ragas (a standard open-source framework for RAG evaluation) integrated with LangSmith, which is already configured in our environment.  

This setup tests key RAG metrics:

*   Faithfulness (Is the answer grounded only in retrieved context?)
*   Answer Relevance (Does the answer address the question?)
*   Context Recall (Did retriever retrieve all necessary context?)
*   Context Precision (Are top retrieved chunks relevant?)

###**Define Test Dataset and Run Pipeline Evaluation**

In [22]:
import time

from ragas import evaluate as ragas_evaluate
from ragas.metrics.collections import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import llm_factory
from google import genai
from langsmith import Client, evaluate as langsmith_evaluate
from datasets import Dataset

# ==========================================
# 1. Top 6 Evaluation Questions & Ground Truths
# ==========================================
eval_questions = [
    "What is the fundamental difference between Artificial Narrow AI (ANI) and Artificial General AI (AGI)?",
    "What were 'AI Winters' in the historical development of AI?",
    "What three key factors drove the modern Deep Learning boom starting in the 2010s?",
    "Differentiate between Overfitting and Underfitting.",
    "Compare Supervised Learning and Unsupervised Learning.",
    "Provide one concrete real-world example for a classification task and one for a clustering task."
]

ground_truths = [
    ["ANI is designed to perform a specific, single task, whereas AGI refers to hypothetical AI with human-level intelligence across a wide range of cognitive tasks."],
    ["AI Winters were historical periods of reduced funding and research activity in AI following overhyped expectations."],
    ["The modern deep learning boom was driven by Big Data, GPUs, and algorithmic advances like Transformers."],
    ["Overfitting occurs when a model learns training noise and fails on unseen data; underfitting occurs when a model is too simple to capture underlying patterns."],
    ["Supervised learning uses labeled data pairs to train models, while unsupervised learning works on unlabeled data to discover hidden patterns."],
    ["Classification: Spam filtering. Clustering: Customer segmentation."]
]

# ==========================================
# 2. Collect Outputs with Rate-Limiting
# ==========================================
answers = []
contexts = []

print("Running predictions for top 6 questions through RAG chain...")
for i, query in enumerate(eval_questions, 1):
    print(f"Processing Question {i}/{len(eval_questions)}...")
    retrieved_docs = vector_db.similarity_search(query, k=4)
    retrieved_texts = [doc.page_content for doc in retrieved_docs]

    response = rag_chain.invoke(query)

    answers.append(response.content if hasattr(response, 'content') else str(response))
    contexts.append(retrieved_texts)

    time.sleep(3)

# ==========================================
# 3. RAGAS Evaluation
# ==========================================
data = {
    "question": eval_questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

ragas_dataset = Dataset.from_dict(data)

eval_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
eval_llm = llm_factory("gemini-1.5-flash", provider="google", client=eval_client)

print("\nRunning RAGAS Evaluation...")
ragas_results = ragas_evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness(),
        answer_relevancy(),
        context_precision(),
        context_recall(),
    ],
    llm=eval_llm
)

df_ragas = ragas_results.to_pandas()
print("\n--- RAGAS Evaluation Results ---")
print(df_ragas[["question", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]])

# ==========================================
# 4. LangSmith Dataset & Evaluation Run
# ==========================================
client = Client()
langsmith_dataset_name = "AI_Primer_Top6_RAG_Evaluation_Dataset"

if not client.has_dataset(dataset_name=langsmith_dataset_name):
    ds = client.create_dataset(
        dataset_name=langsmith_dataset_name,
        description="AI Primer Top 6 RAG test suite."
    )
    for q, gt in zip(eval_questions, ground_truths):
        client.create_example(
            inputs={"question": q},
            outputs={"ground_truth": gt[0]},
            dataset_id=ds.id,
        )

def predict_target(inputs: dict) -> dict:
    q = inputs["question"]
    response = rag_chain.invoke(q)
    return {
        "output": response.content if hasattr(response, 'content') else str(response)
    }

print("\nRunning LangSmith Evaluation Run...")
langsmith_results = langsmith_evaluate(
    predict_target,
    data=langsmith_dataset_name,
    experiment_prefix="AI_Primer_Top6_Experiment"
)


Running predictions for top 6 questions through RAG chain...
Processing Question 1/6...


GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 56.058181992s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '